# Train and compare object detectors

This notebook runs the same workflow as the command-line script. Start with COCO128, then point the configuration at a labeled insect dataset. Set up the Python environment described in [the workflow guide](detection/README.md) before running the cells.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import subprocess
import sys

notebooks = Path.cwd()
if (notebooks / 'Notebooks' / 'detection').is_dir():
    notebooks /= 'Notebooks'
pipeline = notebooks / 'detection' / 'pipeline.py'
assert pipeline.is_file(), 'Open Jupyter from the repository root or Notebooks directory.'

models = ['yolov8n', 'yolo11n', 'mobilenetv3-large']
adapters = {}  # {'ssdlite': notebooks / 'detection/adapters/torchvision_ssdlite.py'}
dataset = None  # Path('/path/to/insect_dataset')
group_by_event = False  # True for images/<event>/ and labels/<event>/
classes = None  # Path('/path/to/classes.json')
epochs = 1
size = 256
batch = 8
device = 'cpu'
seed = 42
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
runs = notebooks.parent / 'detection-runs'

def command(output, dataset, classes, group_by_event=False):
    args = [sys.executable, str(pipeline), '--out', str(output),
            '--models', *models, '--epochs', str(epochs), '--size', str(size),
            '--batch', str(batch), '--device', device, '--seed', str(seed)]
    for name, path in adapters.items():
        args += ['--adapter', f'{name}={path}']
    if dataset is not None:
        args += ['--dataset', str(dataset), '--classes', str(classes)]
    if group_by_event:
        args += ['--group-by-event']
    return args

## Inspect the dataset

Training updates weights, validation selects the checkpoint, and the held-out test split measures the final exports. INT8 calibration uses training images. Check the box counts by class and the annotated previews before continuing. A zero count means that class has no annotated boxes in that split.

In [ ]:
import pandas as pd
from IPython.display import Image, display

inspection = runs / f'{stamp}-inspection'
subprocess.run([*command(inspection, dataset, classes, group_by_event), '--prepare-only'], check=True)
settings = json.loads((inspection / 'run.json').read_text())
print(settings['split_counts'])
display(pd.DataFrame(settings['class_box_counts']).rename_axis('Class'))
for split in ('train', 'val', 'test'):
    print(split)
    display(Image(filename=str(inspection / f'preview-{split}.jpg')))

## Train and export

The one-epoch default exercises the pipeline. Each detector saves a checkpoint and offers one or more precision variants. All variants use the same test images, input size and CPU inference batch size.

In [ ]:
output = runs / stamp
training_dataset = dataset
class_file = classes
if dataset is None or group_by_event:
    training_dataset = Path(settings['arguments']['dataset'])
    class_file = inspection / 'classes.json'
    class_file.write_text(json.dumps(settings['arguments']['classes']))
args = command(output, training_dataset, class_file)
subprocess.run(args, check=True)

## Compare accuracy and inference time

Use mAP50–95 for the main detection comparison. Latency measures the adapter prediction call on this host, including adapter preprocessing and decoding. TFLite variants include a JSON tensor description.

In [ ]:
import pandas as pd

report = json.loads((output / 'comparison.json').read_text())
rows = [{'model': model, 'export': precision,
         'initialization': results.get('initialization'), **metrics}
        for model, results in report.items() for precision, metrics in results.items()
        if isinstance(metrics, dict) and 'mAP50-95' in metrics]
display(pd.DataFrame(rows).set_index(['model', 'export']))
for model in models:
    curve = output / model / 'training' / 'results.png'
    if curve.is_file():
        print(model)
        display(Image(filename=str(curve)))